# Week 5 lab · A week of hourly temperatures

**CM3801 · CMT309/509 · CMT124** · one-hour lab

A weather station records the air temperature **every hour**. One week of readings fits naturally in a 2-D array: **7 rows** (days) × **24 columns** (hours). You will use this week's numpy skills to summarise it: means along an axis, a Boolean mask for frost, anomalies by broadcasting, a faulty reading, and a correlation.

**How this lab works**
* Work through the tasks in order. Each task has an empty cell for your code, followed by a **checkpoint** cell.
* Run the checkpoint after your code. If it prints `✔`, move on. If it raises an `AssertionError`, read the message, fix your code and try again.
* If you are stuck on Task 2, the cell after its checkpoint recreates the results so that you can continue.
* **Question** cells ask you to explain a result in a sentence or two. Write your answer in the text cell (double-click to edit).
* The **Challenge** at the end is optional.

**The data are synthetic**: realistic values for a Cardiff week in November, generated with a fixed seed so that everyone gets the same numbers. They are not Met Office observations.

## The data

Run this cell first. You don't need to understand how the numbers are made: what matters is the result, `temps` (°C) and `humidity` (%), both of shape `(7, 24)`. Row `0` is Monday; column `0` is midnight to 1 am, column `14` is 2 pm.

In [ ]:
import numpy as np

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
hours = np.arange(24)                                   # 0 = midnight to 23 = 11 pm

# Synthetic data: a cold, clear spell in the middle of a November week in Cardiff
rng = np.random.default_rng(2026)
day_level = np.array([9.5, 8.0, 5.5, 2.5, 1.5, 4.5, 8.0])   # typical level of each day, °C
day_swing = np.array([2.0, 2.5, 3.5, 4.5, 4.0, 3.0, 2.0])   # how far it warms by day / cools by night
daily_cycle = -np.cos(2 * np.pi * (hours - 4) / 24)         # coldest around 4 am, warmest around 4 pm
temps = day_level.reshape(7, 1) + day_swing.reshape(7, 1) * daily_cycle + rng.normal(0, 0.4, size=(7, 24))
temps = temps.round(1)                                      # hourly air temperature, °C

humidity = 88 - 2.5 * (temps - temps.mean()) + rng.normal(0, 4, size=(7, 24))
humidity = np.clip(humidity, 50, 100).round(0)              # relative humidity, %

print(temps.shape, humidity.shape)
print(temps[0, :6])

## Task 1 · Know your array

Create:
* `n_days` and `n_hours`: the two sizes of `temps` (use its `shape`, don't type the numbers);
* `mon_2pm`: Monday's temperature at 2 pm (row 0, column 14);
* `coldest` and `warmest`: the lowest and highest reading of the whole week.

Print `temps.dtype` too: what type are the values?

In [ ]:
# STUDENT TASK
# Task: n_days, n_hours, mon_2pm, coldest, warmest
# Hint: tuple unpacking works on a shape: rows, cols = arr.shape
# your code here


In [ ]:
# Checkpoint 1
assert (n_days, n_hours) == (7, 24), "n_days, n_hours: unpack temps.shape"
assert np.isclose(mon_2pm, 11.3), "mon_2pm: row 0 (Monday), column 14 (2 pm)"
assert np.isclose(coldest, -3.3) and np.isclose(warmest, 11.7), "coldest/warmest: the min and max of the whole array"
print("✔ Task 1")

## Task 2 · Daily and hourly means with `axis`

1. `daily_mean`: the mean temperature of **each day** (one value per row). Its shape should be `(7,)`.
2. `hourly_mean`: the mean temperature at **each hour of the day**, over the week (one value per column). Shape `(24,)`.
3. `coldest_day`: the **name** of the day with the lowest daily mean. *Hint:* `argmin()` gives a position; use it on `days`.

In [ ]:
# STUDENT TASK
# Task: daily_mean, hourly_mean, coldest_day
# Hint: axis=1 goes across the columns (one result per row); axis=0 goes down the rows
# your code here


In [ ]:
# Checkpoint 2
assert daily_mean.shape == (7,), "daily_mean should have one value per day: which axis runs across the hours?"
assert hourly_mean.shape == (24,), "hourly_mean should have one value per hour"
assert np.isclose(daily_mean[0], 9.47, atol=0.01) and np.isclose(daily_mean[4], 1.48, atol=0.01), "daily_mean: check the axis"
assert np.isclose(hourly_mean[4], 2.41, atol=0.01) and np.isclose(hourly_mean[16], 8.67, atol=0.01), "hourly_mean: check the axis"
assert coldest_day == "Fri", "coldest_day: days[...] at the position of the smallest daily mean"
print("✔ Task 2")

In [ ]:
# If your result does not match the checkpoint, run this cell to continue.
daily_mean = temps.mean(axis=1)
hourly_mean = temps.mean(axis=0)

## Task 3 · Frost hours with a Boolean mask

An air temperature below 0 °C means a frost.
1. `frost`: a Boolean array, `True` where the temperature is below zero. What shape do you expect?
2. `frost_hours`: the total number of frost hours in the week. *Hint:* `True` counts as 1.
3. `frost_per_day`: the number of frost hours **on each day** (7 values).
4. `frost_temps`: all the below-zero readings, selected with the mask.
5. `thu_frost_hours`: **which hours** (from `hours`) had a frost on Thursday (row 3). *Hint:* use row 3 of the mask as an index into `hours`.

In [ ]:
# STUDENT TASK
# Task: frost, frost_hours, frost_per_day, frost_temps, thu_frost_hours
# your code here


In [ ]:
# Checkpoint 3
assert frost.shape == (7, 24) and frost.dtype == bool, "frost: compare temps with 0"
assert frost_hours == 15, "frost_hours: sum the Boolean array"
assert list(frost_per_day) == [0, 0, 0, 7, 8, 0, 0], "frost_per_day: sum along the hours (one value per day)"
assert len(frost_temps) == 15 and (frost_temps < 0).all(), "frost_temps: index temps with the mask"
assert list(thu_frost_hours) == [1, 2, 3, 4, 5, 6, 7], "thu_frost_hours: hours[frost[3]]"
print("✔ Task 3")

## Task 4 · Anomalies by broadcasting

An **anomaly** is how far a reading is from a reference value. Compute two kinds, **without loops**:
1. `day_anom`: each reading minus **its own day's** mean. Shape `(7, 24)`.
2. `hour_anom`: each reading minus the week's mean for **that hour of the day** (`hourly_mean`). Shape `(7, 24)`.

*Hint:* think about the shapes first. `temps` is `(7, 24)`; `daily_mean` is `(7,)` and `hourly_mean` is `(24,)`. One of them broadcasts as it is; the other needs `.reshape(...)` into a column.

In [ ]:
# STUDENT TASK
# Task: day_anom and hour_anom (broadcasting)
# your code here


In [ ]:
# Checkpoint 4
assert day_anom.shape == (7, 24) and hour_anom.shape == (7, 24), "both anomaly arrays should have the shape of temps"
assert np.allclose(day_anom.mean(axis=1), 0), "day_anom: each day's anomalies should average to 0 (subtract that day's mean)"
assert np.allclose(hour_anom.mean(axis=0), 0), "hour_anom: each hour's anomalies should average to 0 over the week"
assert np.isclose(day_anom[3, 14], temps[3, 14] - temps[3].mean()), "day_anom: reshape daily_mean to (7, 1)"
print("✔ Task 4")

**Question 4.** Why does `temps - daily_mean` (without the reshape) raise an error, while `temps - hourly_mean` works? Use the broadcasting rules in your answer.

*Your answer:*

## Task 5 · A faulty sensor: mean or median?

On Wednesday at 1 pm the sensor glitched and logged **85.0 °C**.
1. Create `spiky`: a **copy** of `temps` in which row 2, column 13 is `85.0`. `temps` itself must not change.
2. For Wednesday (row 2), compute `wed_mean_clean`, `wed_median_clean` (from `temps`) and `wed_mean_spiky`, `wed_median_spiky` (from `spiky`).

In [ ]:
# STUDENT TASK
# Task: spiky (a copy with one bad value) and Wednesday's mean/median, clean and spiky
# your code here


In [ ]:
# Checkpoint 5
assert spiky[2, 13] == 85.0 and temps[2, 13] != 85.0, "spiky must be a copy: temps should not contain 85.0"
assert np.isclose(wed_mean_clean, 5.52, atol=0.01) and np.isclose(wed_mean_spiky, 8.74, atol=0.01), "Wednesday's means: row 2"
assert np.isclose(wed_median_clean, 5.4) and np.isclose(wed_median_spiky, 5.4), "Wednesday's medians: np.median of row 2"
print("✔ Task 5")

**Question 5.** By how much did the single bad reading move Wednesday's mean, and its median? If you had to report "Wednesday's typical temperature" from data that might contain glitches, which would you use, and what is the better long-term fix? And if a person had read a thermometer by hand instead, which problems would you expect then?

*Your answer:*

## Task 6 · Do temperature and humidity move together?

1. Flatten both arrays into one long series of 168 hourly values each: `t_flat` and `h_flat`. *Hint:* `.reshape(168)` (or `.reshape(7 * 24)`).
2. `cov_th`: the covariance of temperature and humidity (`np.cov`, then pick the pair's value).
3. `r_th`: their correlation coefficient (`np.corrcoef`), a single number.

In [ ]:
# STUDENT TASK
# Task: t_flat, h_flat, cov_th, r_th
# Hint: np.cov and np.corrcoef return a 2 x 2 matrix; the pair's value is at [0, 1]
# your code here


In [ ]:
# Checkpoint 6
assert t_flat.shape == (168,) and h_flat.shape == (168,), "flatten with .reshape(168)"
assert np.isclose(cov_th, -26.51, atol=0.01), "cov_th: np.cov(t_flat, h_flat)[0, 1]"
assert np.isclose(r_th, -0.891, atol=0.001), "r_th: np.corrcoef(t_flat, h_flat)[0, 1]"
print("✔ Task 6")

**Question 6.** What do the sign and the size of `r_th` tell you? Why is the correlation easier to interpret than the covariance? And does the correlation prove that warming *causes* the air to become drier?

*Your answer:*

## Challenge (optional) · Clean the spike and find the coldest hour

1. In `spiky`, replace every reading above 40 °C with Wednesday's **median** (from `temps`), using a Boolean mask on the left of `=`. Check that Wednesday's mean is back close to the clean value.
2. Using `hourly_mean`, find the **hour of the day** that is coldest on average across the week, and the warmest.
3. Which day had the **largest range** (warmest minus coldest hour)? *Hint:* `.max(axis=1) - .min(axis=1)`.

In [ ]:
# STUDENT TASK
# Task: clean the spike; coldest/warmest hour on average; day with the largest range
# your code here


## Wrap-up

You treated a week of readings as one 2-D array: `axis` gave daily and hourly summaries, a Boolean mask counted frost hours, broadcasting turned readings into anomalies without a loop, the median survived a faulty sensor that fooled the mean, and a correlation measured how two variables move together. Next week, pandas will give the rows and columns **names** (days, hours, stations) on top of exactly these arrays.